## Tymebank Review Scraper - Complete Dataset


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime
import json
import warnings
from tqdm import tqdm
import re
import os

warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully')
print(f'Pandas version: {pd.__version__}')
print(f'Start time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

In [ ]:
# ============================================================================
# CONFIGURATION SECTION
# ============================================================================

# Target URL
BASE_URL = 'https://www.hellopeter.com/tymebank'

# Scraping parameters
CONFIG = {
    'base_url': BASE_URL,
    'timeout': 10,  # Request timeout in seconds
    'delay': 2,  # Delay between requests in seconds
    'max_retries': 3,  # Maximum retries per page
    'retry_delay': 5,  # Delay before retry in seconds
    'estimated_reviews': 3859,  # Estimated total reviews
    'reviews_per_page': 50,  # Reviews typically shown per page
}

# User-Agent to avoid being blocked
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1'
}

print('📋 CONFIGURATION')
print(f"Target URL: {CONFIG['base_url']}")
print(f"Estimated total reviews: {CONFIG['estimated_reviews']:,}")
print(f"Estimated pages (50 per page): {CONFIG['estimated_reviews'] // CONFIG['reviews_per_page'] + 1}")
print(f"Request timeout: {CONFIG['timeout']}s")
print(f"Delay between requests: {CONFIG['delay']}s")
print(f"Max retries: {CONFIG['max_retries']}")

In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def fetch_page(url, session, max_retries=3):
    """
    Fetch a single page with retry logic
    
    Parameters:
    -----------
    url : str
        URL to fetch
    session : requests.Session
        Requests session object
    max_retries : int
        Maximum number of retries
    
    Returns:
    --------
    BeautifulSoup or None : Parsed HTML or None if failed
    """
    for attempt in range(max_retries):
        try:
            response = session.get(url, headers=HEADERS, timeout=CONFIG['timeout'])
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                time.sleep(CONFIG['retry_delay'])
            else:
                return None
    return None


def extract_review_from_element(element):
    """
    Extract review data from a review element
    
    Parameters:
    -----------
    element : BeautifulSoup tag
        Review element
    
    Returns:
    --------
    dict : Review data or None if extraction failed
    """
    try:
        # Extract reviewer name
        reviewer_elem = element.find('span', class_=['reviewer-name', 'customer-name', 'author'])
        reviewer_name = reviewer_elem.text.strip() if reviewer_elem else 'Anonymous'
        
        # Extract review title
        title_elem = element.find(['h3', 'strong'], class_=['review-title', 'review-heading'])
        review_title = title_elem.text.strip() if title_elem else 'No Title'
        
        # Extract review text
        text_elem = element.find('p', class_=['review-text', 'review-content', 'comment-text'])
        if not text_elem:
            text_elem = element.find('div', class_=['review-body', 'review-content'])
        review_text = text_elem.text.strip() if text_elem else 'No Text'
        
        # Extract rating
        rating_elem = element.find('div', class_=['rating', 'star-rating'])
        if not rating_elem:
            rating_elem = element.find('span', class_=['rating', 'stars'])
        
        rating = None
        if rating_elem:
            rating_text = rating_elem.get('data-rating')
            if not rating_text:
                rating_text = rating_elem.text
            try:
                rating = int(re.search(r'\d', rating_text).group())
            except:
                rating = None
        
        # Extract review date
        date_elem = element.find('span', class_=['review-date', 'date', 'time'])
        if not date_elem:
            date_elem = element.find('time')
        review_date = date_elem.text.strip() if date_elem else 'Unknown'
        
        return {
            'reviewer_name': reviewer_name,
            'review_title': review_title,
            'review_text': review_text,
            'rating': rating,
            'review_date': review_date,
            'scrape_timestamp': datetime.now().isoformat(),
            'page_url': None  # Will be set by caller
        }
    except Exception as e:
        return None


def get_total_pages(soup):
    """
    Extract total number of pages from pagination
    
    Parameters:
    -----------
    soup : BeautifulSoup
        Parsed HTML
    
    Returns:
    --------
    int : Total pages or None if not found
    """
    try:
        # Try multiple pagination selectors
        pagination = soup.find('div', class_='pagination')
        if pagination:
            last_page = pagination.find_all('a')[-1]
            if last_page:
                page_num = int(re.search(r'\d+', last_page.text).group())
                return page_num
    except:
        pass
    return None

print('✅ Helper functions defined')

In [ ]:
# ============================================================================
# MAIN SCRAPER FUNCTION
# ============================================================================

def scrape_all_reviews(base_url, max_pages=None):
    """
    Scrape all reviews from HelloPeter
    
    Parameters:
    -----------
    base_url : str
        Base URL for the company
    max_pages : int
        Maximum pages to scrape (None = all)
    
    Returns:
    --------
    list : List of review dictionaries
    """
    
    all_reviews = []
    session = requests.Session()
    
    print('🔍 STARTING WEB SCRAPER')
    print(f'Target: {base_url}')
    print()
    
    # Fetch first page to determine total pages
    print('📥 Fetching first page to determine total pages...')
    soup = fetch_page(base_url, session)
    
    if soup is None:
        print('❌ Failed to fetch first page')
        return []
    
    total_pages = get_total_pages(soup)
    if total_pages is None:
        # Estimate based on config
        total_pages = CONFIG['estimated_reviews'] // CONFIG['reviews_per_page']
    
    if max_pages:
        total_pages = min(total_pages, max_pages)
    
    print(f'✅ Determined total pages: {total_pages}')
    print(f'⏱️ Estimated time: {total_pages * CONFIG["delay"]} seconds')
    print()
    
    # Scrape all pages
    for page_num in tqdm(range(1, total_pages + 1), desc='Scraping reviews', unit='page'):
        # Construct page URL
        if page_num == 1:
            page_url = base_url
        else:
            page_url = f'{base_url}?page={page_num}'
        
        # Fetch page
        soup = fetch_page(page_url, session)
        
        if soup is None:
            print(f'\n⚠️ Failed to fetch page {page_num}')
            continue
        
        # Find review containers (try multiple selectors)
        review_containers = soup.find_all('div', class_='review-item')
        if not review_containers:
            review_containers = soup.find_all('article', class_='review')
        if not review_containers:
            review_containers = soup.find_all('div', class_=['review', 'review-container'])
        
        # Extract reviews from page
        for review_elem in review_containers:
            review_data = extract_review_from_element(review_elem)
            if review_data:
                review_data['page_url'] = page_url
                review_data['page_number'] = page_num
                all_reviews.append(review_data)
        
        # Respectful delay between requests
        time.sleep(CONFIG['delay'])
    
    return all_reviews

print('✅ Scraper function defined')

In [ ]:
# ============================================================================
# EXECUTE SCRAPER
# ============================================================================

print('='*80)
print('STARTING SCRAPING PROCESS')
print('='*80)
print()

start_time = datetime.now()

# Run scraper
# NOTE: Set max_pages=None to scrape all pages, or max_pages=5 for testing
reviews = scrape_all_reviews(CONFIG['base_url'], max_pages=None)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print()
print('='*80)
print('SCRAPING COMPLETE')
print('='*80)
print(f'Total reviews collected: {len(reviews):,}')
print(f'Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)')
print(f'Average speed: {len(reviews)/duration:.1f} reviews/second')

In [ ]:
# ============================================================================
# DATA VALIDATION & CLEANING
# ============================================================================

print('\n📊 DATA VALIDATION')
print('-'*80)

# Create DataFrame
df = pd.DataFrame(reviews)

print(f'\nDataFrame shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()

# Data quality checks
print('📋 DATA QUALITY CHECKS:')
print(f'  Total rows: {len(df)}')
print(f'  Null reviewer names: {df["reviewer_name"].isna().sum()}')
print(f'  Null review text: {df["review_text"].isna().sum()}')
print(f'  Null ratings: {df["rating"].isna().sum()}')
print(f'  Reviews with empty text: {(df["review_text"] == "No Text").sum()}')
print()

# Rating distribution
print('⭐ RATING DISTRIBUTION:')
if df['rating'].notna().sum() > 0:
    rating_dist = df['rating'].value_counts().sort_index()
    for rating, count in rating_dist.items():
        pct = (count / len(df)) * 100
        print(f'  {rating} stars: {count:,} reviews ({pct:.1f}%)')
else:
    print('  No rating data available')
print()

# Text statistics
print('📝 TEXT STATISTICS:')
df['text_length'] = df['review_text'].str.len()
df['word_count'] = df['review_text'].str.split().str.len()
print(f'  Average review length: {df["text_length"].mean():.0f} characters')
print(f'  Average review length: {df["word_count"].mean():.1f} words')
print(f'  Shortest review: {df["text_length"].min()} characters')
print(f'  Longest review: {df["text_length"].max()} characters')
print()

# Handle missing values
print('🔧 CLEANING DATA:')
df['reviewer_name'] = df['reviewer_name'].fillna('Anonymous')
df['review_text'] = df['review_text'].fillna('')
df['review_date'] = df['review_date'].fillna('Unknown')
print('  ✓ Filled missing reviewer names with "Anonymous"')
print('  ✓ Filled missing review text with empty string')
print('  ✓ Filled missing dates with "Unknown"')
print()

print('✅ Data validation complete')

In [ ]:
# ============================================================================
# EXPORT TO CSV
# ============================================================================

print('\n💾 EXPORTING TO CSV')
print('-'*80)

# Select columns for export
export_columns = [
    'reviewer_name',
    'review_title',
    'review_text',
    'rating',
    'review_date',
    'page_number',
    'page_url',
    'scrape_timestamp'
]

# Ensure output directory exists
output_dir = '/mnt/user-data/outputs/'
os.makedirs(output_dir, exist_ok=True)

# Export filename
filename = 'tymebank_all_reviews_3859.csv'
filepath = os.path.join(output_dir, filename)

# Export to CSV
df_export = df[export_columns].copy()
df_export.to_csv(filepath, index=False, encoding='utf-8')

print(f'✅ Exported to: {filepath}')
print(f'   File size: {os.path.getsize(filepath) / (1024*1024):.2f} MB')
print(f'   Total rows: {len(df_export):,}')
print(f'   Total columns: {len(df_export.columns)}')
print()

# Display sample
print('📋 SAMPLE DATA (first 3 rows):')
print()
print(df_export.head(3).to_string())

In [ ]:
# ============================================================================
# SUMMARY REPORT
# ============================================================================

print('\n' + '='*80)
print('SCRAPING SUMMARY REPORT')
print('='*80)

summary = f"""
📊 PROJECT SUMMARY

Target Website: HelloPeter.com
Company: TymeBank
URL: {CONFIG['base_url']}

📈 RESULTS

Total Reviews Scraped: {len(df):,}
Expected Reviews: {CONFIG['estimated_reviews']:,}
Completion: {(len(df) / CONFIG['estimated_reviews'] * 100):.1f}%

⏱️ PERFORMANCE

Start Time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}
End Time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}
Total Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)
Average Speed: {len(df)/duration:.2f} reviews/second

📝 DATA STATISTICS

Average Review Length: {df['text_length'].mean():.0f} characters
Average Review Length: {df['word_count'].mean():.1f} words
Reviewer Names Filled: {(df['reviewer_name'] == 'Anonymous').sum():,}
Ratings Available: {df['rating'].notna().sum():,}

⭐ RATING BREAKDOWN
"""

print(summary)

if df['rating'].notna().sum() > 0:
    rating_dist = df['rating'].value_counts().sort_index()
    for rating, count in rating_dist.items():
        pct = (count / len(df)) * 100
        bar = '█' * int(pct / 2)
        print(f'{rating} stars: {count:6,} reviews ({pct:5.1f}%) {bar}')

print(f'\n💾 OUTPUT FILE')
print(f'\nFile: {filename}')
print(f'Path: {filepath}')
print(f'Size: {os.path.getsize(filepath) / (1024*1024):.2f} MB')
print(f'Rows: {len(df_export):,}')
print(f'Columns: {len(df_export.columns)}')
print()
print('✅ SCRAPING COMPLETE - Ready for analysis!')

In [ ]:
# ============================================================================
# RELOAD AND EXPLORE SAVED DATA
# ============================================================================

# This cell can be run separately to reload the saved CSV

print('\n📂 LOADING SAVED DATA')
print('-'*80)

# Load CSV
df_loaded = pd.read_csv(filepath)

print(f'✅ Loaded {len(df_loaded):,} reviews from {filename}')
print()
print('📊 DATA INFO:')
print(f'  Shape: {df_loaded.shape}')
print(f'  Columns: {list(df_loaded.columns)}')
print(f'  Memory usage: {df_loaded.memory_usage(deep=True).sum() / (1024*1024):.2f} MB')
print()
print('🔍 COLUMN DETAILS:')
print()
print(df_loaded.info())
print()
print('📊 DESCRIPTIVE STATISTICS:')
print()
print(df_loaded.describe())

## Next Steps

Now that you have all 3,859 TymeBank reviews, you can:

### 1. Data Exploration
```python
# Load the data
df = pd.read_csv('tymebank_all_reviews_3859.csv')

# Explore ratings
print(df['rating'].value_counts())

# Review text analysis
print(df['review_text'].str.len().describe())
```

### 2. Data Cleaning & Preprocessing
- Remove URLs, emails, special characters
- Normalize text (lowercase, remove whitespace)
- Handle missing values

### 3. NLP & Sentiment Analysis
- Tokenization and lemmatization
- VADER, TextBlob, or Transformer sentiment analysis
- Sentiment correlation with ratings

### 4. Statistical Analysis
- Rating distribution analysis
- Sentiment trends over time
- Key themes and topics

### 5. Visualization
- Create plots and charts
- Generate dashboard
- Export results

---

## Important Notes

### Respect the Website
- Use a reasonable delay between requests (default: 2 seconds)
- Don't scrape during peak hours
- Check the website's robots.txt and Terms of Service
- Consider using the site's API if available

### Data Quality
- Check for and handle missing values
- Validate data types (ratings should be 1-5)
- Remove duplicates if any
- Verify sample reviews manually

### Large Dataset Handling
- With ~3,859 reviews, the CSV will be ~2-3 MB
- Use `chunksize` parameter for large data processing
- Consider using efficient data types (e.g., category for ratings)

### Troubleshooting
- If getting 403/blocked, try:
  - Increase delay between requests
  - Use different User-Agent strings
  - Consider using rotating proxies
- If getting incomplete data:
  - Check selector names (they may have changed)
  - Print HTML to debug
  - Use browser developer tools to inspect

---

**Course:** Postgraduate Diploma in Data Science - Finance Elective (SS1-DFS182)  
**Institution:** STADIO School of Information Technology (SOIT)  
**Date:** October 2, 2026